In [1]:
from pathlib import Path

ordner = Path.home() / "Desktop" / "Modul 5" / "Aufgaben" / "Woche_3"

skript = ordner / "mini_scheduler.py"

print(skript.read_text(encoding="utf-8"))

"""Mini-Scheduler — Übungsgerüst zu Tag 13 (Trigger & Scheduling).

Ein Scheduler in 30 Zeilen: eine Endlosschleife, die eure Pipeline
regelmäßig startet und jeden Lauf protokolliert.

ACHTUNG — nur zum Lernen: Dieses Skript zeigt, WAS ein Scheduler tut.
Im echten Betrieb übernimmt cron (oder ein Orchestrator) das Timing —
warum, steht in der Vorlesung (die vier Fallen).

Aufgabe: die vier TODOs vervollständigen, dann starten mit
    python3 mini_scheduler.py
und nach mindestens 3 Läufen mit Strg+C stoppen.
"""

import time
from datetime import datetime
from pathlib import Path

INTERVALL_SEKUNDEN = 10          # didaktisch kurz — cron kann frühestens 1×/Minute
PROTOKOLL = Path("mini_protokoll.csv")


def pipeline_lauf() -> int:
    """Simulierter Pipeline-Lauf (gestern: der Ingest aus der Mock-API).

    Gibt die Zahl der 'geholten Zeilen' zurück.
    """
    # TODO 1: Simuliert hier einen Lauf — z. B. 2 Sekunden "Arbeit"
    #         (time.sleep) und ein fester Rückgabewert genügt.


In [2]:
import pandas as pd

protokoll = pd.read_csv(
    ordner / "mini_protokoll.csv",
    sep=";",
    parse_dates=["start", "ende"]
)

protokoll["dauer_sekunden"] = (
    protokoll["ende"] - protokoll["start"]
).dt.total_seconds()

protokoll["startabstand_sekunden"] = (
    protokoll["start"].diff()
).dt.total_seconds()

display(protokoll)

,start,ende,zeilen,dauer_sekunden,startabstand_sekunden
0,2026-11-16 11:51:27.033332,2026-11-16 11:51:29.038401,111,2.005069,NaN
1,2026-11-16 11:51:39.041724,2026-11-16 11:51:41.046772,111,2.005048,12.008392
2,2026-11-16 11:51:51.049820,2026-11-16 11:51:53.053870,111,2.004050,12.008096
3,2026-11-16 11:52:03.056701,2026-11-16 11:52:05.059666,111,2.002965,12.006881


### Reflexion: Mini-Scheduler vs. Cron

Beim Test des Mini-Schedulers wurden vier erfolgreiche Pipeline-Läufe protokolliert. Jeder Lauf verarbeitete 111 Zeilen und dauerte ungefähr zwei Sekunden.

Obwohl das Intervall auf zehn Sekunden eingestellt war, betrug der tatsächliche Abstand zwischen den Startzeitpunkten etwa zwölf Sekunden.

Der Grund ist, dass `time.sleep(10)` erst nach Abschluss der Verarbeitung ausgeführt wird. Dadurch addieren sich Laufzeit und Wartezeit. Kleine zusätzliche Verzögerungen können sich ebenfalls aufsummieren.

Cron startet Aufgaben dagegen zu festgelegten Zeitpunkten. Die geplanten Startzeiten verschieben sich nicht durch die Laufzeit vorheriger Ausführungen. Bei längeren Laufzeiten muss allerdings verhindert werden, dass sich mehrere Prozesse unerwünscht überschneiden.

In [3]:
laufdaten = pd.read_csv(ordner / "lauf_protokoll.csv")

print("Spalten:", laufdaten.columns.tolist())
print("Anzahl Einträge:", len(laufdaten))

display(laufdaten.head())

Spalten: ['start_utc', 'ende_utc', 'status', 'zeilen_geholt']
Anzahl Einträge: 87


,start_utc,ende_utc,status,zeilen_geholt
0,2026-03-09 00:00:00,2026-03-09 00:05:35,ok,82
1,2026-03-09 01:00:00,2026-03-09 01:05:16,ok,82
2,2026-03-09 02:00:00,2026-03-09 02:02:16,ok,79
3,2026-03-09 03:00:00,2026-03-09 03:05:46,ok,45
4,2026-03-09 04:00:00,2026-03-09 04:02:40,ok,34


In [4]:
print("Statusverteilung:")
print(laufdaten["status"].value_counts())

print("\nFehlerhafte Läufe:")
display(laufdaten[laufdaten["status"] != "ok"])

Statusverteilung:
status
ok        84
fehler     3
Name: count, dtype: int64

Fehlerhafte Läufe:


,start_utc,ende_utc,status,zeilen_geholt
14,2026-03-09 14:00:00,2026-03-09 14:03:22,fehler,0
66,2026-03-11 22:00:00,2026-03-11 22:03:05,fehler,0
71,2026-03-12 03:00:00,2026-03-12 03:02:38,fehler,0


In [5]:
laufdaten["start_utc"] = pd.to_datetime(laufdaten["start_utc"])

laufdaten = laufdaten.sort_values("start_utc").copy()

laufdaten["abstand_stunden"] = (
    laufdaten["start_utc"].diff()
).dt.total_seconds() / 3600

print("Abstände zwischen den Pipeline-Starts:")
print(laufdaten["abstand_stunden"].value_counts().sort_index())

print("\nAuffällige Abstände über einer Stunde:")
display(laufdaten[laufdaten["abstand_stunden"] > 1])

Abstände zwischen den Pipeline-Starts:
abstand_stunden
1.0    85
5.0     1
Name: count, dtype: int64

Auffällige Abstände über einer Stunde:


,start_utc,ende_utc,status,zeilen_geholt,abstand_stunden
26,2026-03-10 06:00:00,2026-03-10 06:05:58,ok,69,5.0


In [6]:
erwartete_starts = pd.date_range(
    start=laufdaten["start_utc"].min(),
    end=laufdaten["start_utc"].max(),
    freq="h"
)

tatsaechliche_starts = pd.DatetimeIndex(laufdaten["start_utc"])

fehlende_starts = erwartete_starts.difference(tatsaechliche_starts)

print("Erwartete Starts:", len(erwartete_starts))
print("Protokollierte Starts:", len(tatsaechliche_starts))
print("Fehlende Starts:", len(fehlende_starts))

print("\nFehlende Zeitpunkte:")
print(fehlende_starts)

Erwartete Starts: 91
Protokollierte Starts: 87
Fehlende Starts: 4

Fehlende Zeitpunkte:
DatetimeIndex(['2026-03-10 02:00:00', '2026-03-10 03:00:00',
               '2026-03-10 04:00:00', '2026-03-10 05:00:00'],
              dtype='datetime64[us]', freq='h')


In [7]:
laufdaten["ende_utc"] = pd.to_datetime(laufdaten["ende_utc"])

laufdaten["laufzeit_minuten"] = (
    laufdaten["ende_utc"] - laufdaten["start_utc"]
).dt.total_seconds() / 60

print("Laufzeit-Statistik in Minuten:")
print(laufdaten["laufzeit_minuten"].describe())

print("\nDie 5 längsten Pipeline-Läufe:")
display(
    laufdaten.nlargest(5, "laufzeit_minuten")[
        ["start_utc", "status", "zeilen_geholt", "laufzeit_minuten"]
    ]
)

Laufzeit-Statistik in Minuten:
count    87.000000
mean      5.018391
std       7.568681
min       2.116667
25%       3.258333
50%       4.400000
75%       5.150000
max      74.000000
Name: laufzeit_minuten, dtype: float64

Die 5 längsten Pipeline-Läufe:


,start_utc,status,zeilen_geholt,laufzeit_minuten
53,2026-03-11 09:00:00,ok,52,74.000000
26,2026-03-10 06:00:00,ok,69,5.966667
23,2026-03-09 23:00:00,ok,53,5.916667
29,2026-03-10 09:00:00,ok,47,5.866667
40,2026-03-10 20:00:00,ok,39,5.833333


In [8]:
auswahl = laufdaten[
    (laufdaten["start_utc"] >= "2026-03-11 08:00:00") &
    (laufdaten["start_utc"] <= "2026-03-11 11:00:00")
]

display(
    auswahl[
        ["start_utc", "ende_utc", "status", "laufzeit_minuten"]
    ]
)

,start_utc,ende_utc,status,laufzeit_minuten
52,2026-03-11 08:00:00,2026-03-11 08:05:29,ok,5.483333
53,2026-03-11 09:00:00,2026-03-11 10:14:00,ok,74.000000
54,2026-03-11 10:00:00,2026-03-11 10:05:09,ok,5.150000
55,2026-03-11 11:00:00,2026-03-11 11:05:45,ok,5.750000


In [9]:
laufdaten["vorheriges_ende"] = laufdaten["ende_utc"].shift(1)

ueberschneidungen = laufdaten[
    laufdaten["start_utc"] < laufdaten["vorheriges_ende"]
]

print("Gefundene Überschneidungen:", len(ueberschneidungen))

display(
    ueberschneidungen[
        ["start_utc", "ende_utc", "vorheriges_ende"]
    ]
)

Gefundene Überschneidungen: 1


,start_utc,ende_utc,vorheriges_ende
54,2026-03-11 10:00:00,2026-03-11 10:05:09,2026-03-11 10:14:00


### Fehleranalyse – Laufprotokoll

Bei der Analyse von `lauf_protokoll.csv` wurden 87 Pipeline-Läufe untersucht.

**1. Fehlgeschlagene Läufe**

84 Läufe waren erfolgreich, 3 hatten den Status `fehler`. Bei allen drei Fehlern wurden 0 Zeilen geholt.

**2. Fehlende Ausführungen**

Bei einem angenommenen stündlichen Zeitplan wären 91 Starts zu erwarten gewesen. Tatsächlich wurden nur 87 protokolliert.

Am 10.03.2026 fehlen die Starts um 02:00, 03:00, 04:00 und 05:00 Uhr (UTC).

**3. Ungewöhnliche Laufzeit**

Am 11.03.2026 dauerte ein Pipeline-Lauf 74 Minuten. Die übrigen Läufe dauerten höchstens etwa 6 Minuten.

**4. Überschneidung**

Der Lauf von 09:00 bis 10:14 Uhr überschnitt sich mit dem Lauf von 10:00 bis 10:05:09 Uhr.

**Fazit und mögliche Verbesserungen**

Für einen zuverlässigen Betrieb sollten fehlgeschlagene und ausgefallene Starts überwacht werden. Ein Locking-Mechanismus könnte verhindern, dass mehrere Pipeline-Läufe gleichzeitig dieselben Daten bearbeiten. Außerdem sollten ungewöhnlich lange Laufzeiten automatisch gemeldet werden.

Die tatsächlichen Ursachen der Fehler und fehlenden Starts lassen sich aus dem vorliegenden Protokoll allein nicht bestimmen.

### Cron-Ausdrücke – Übungsübersicht

Ein Cron-Ausdruck besteht aus fünf Feldern:

`Minute Stunde Tag Monat Wochentag`

| Cron-Ausdruck | Bedeutung |
|---|---|
| `0 7 * * 1-5` | Montag bis Freitag um 07:00 Uhr |
| `*/15 8-18 * * 1-5` | Montag bis Freitag alle 15 Minuten von 08:00 bis 18:45 Uhr |
| `30 3 * * 0` | Jeden Sonntag um 03:30 Uhr |
| `15 22 * * *` | Täglich um 22:15 Uhr |
| `45 6 * * 1,3,5` | Montag, Mittwoch und Freitag um 06:45 Uhr |
| `0 12 * * 6,0` | Samstag und Sonntag um 12:00 Uhr |
| `0 */4 * * *` | Täglich alle vier Stunden |

**Wichtige Erkenntnisse:**

- `*` steht für alle möglichen Werte.
- `*/15` bedeutet alle 15 Einheiten.
- `1-5` bezeichnet einen zusammenhängenden Bereich.
- `1,3,5` bezeichnet einzelne ausgewählte Werte.

Cron eignet sich für zeitgesteuerte Pipeline-Ausführungen. Bei längeren Laufzeiten müssen mögliche Überschneidungen berücksichtigt werden.